In [1]:
import os
import time
import random
import zipfile
import json

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    convnext_tiny,
    ConvNeXt_Tiny_Weights
)

from sklearn.metrics import classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
ZIP_PATH = "/content/drive/MyDrive/Skin Disease Detection/Dataset/dermnet.zip"

TRAIN_DIR = "/content/train"
TEST_DIR = "/content/test"

start = time.time()

if not (os.path.exists(TRAIN_DIR) and os.path.exists(TEST_DIR)):
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall("/content")
else:
    print("Dataset already extracted.")

print(f"Ready in {time.time() - start:.1f} sec")
print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists :", os.path.exists(TEST_DIR))

Extracting dataset...
Ready in 42.4 sec
Train exists: True
Test exists : True


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [5]:
IMG_SIZE = 384
BATCH_SIZE = 8
ACCUM_STEPS = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

print("Transforms ready.")
print("Image size :", IMG_SIZE)
print("Batch size :", BATCH_SIZE)
print("Accum steps:", ACCUM_STEPS)

Transforms ready.
Image size : 384
Batch size : 8
Accum steps: 2


In [6]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

print("Classes:", num_classes)
print("Train images:", len(train_dataset))
print("Test images :", len(test_dataset))
print("Batch size  :", BATCH_SIZE)
print("Accum steps :", ACCUM_STEPS)

Classes: 23
Train images: 15557
Test images : 4002
Batch size  : 8
Accum steps : 2


In [7]:
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = convnext_tiny(weights=weights)

in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, num_classes)

model = model.to(device)

print(model.classifier)
print(f"Total Parameters     : {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable Parameters : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 139MB/s]


Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=23, bias=True)
)
Total Parameters     : 27,837,815
Trainable Parameters : 27,837,815


In [8]:
# ==========================================================
# Cell 8 : Stage 1 - Freeze Backbone
# ==========================================================

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[2].parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Stage 1 setup complete")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Total Parameters     : {total_params:,}")

Stage 1 setup complete
Trainable Parameters : 17,687
Total Parameters     : 27,837,815


In [9]:
# ==========================================================
# Cell 9 : Stage 1 Optimizer + Scheduler
# ==========================================================

EPOCHS_STAGE1 = 3
LR_STAGE1 = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE1,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS_STAGE1
)

print("Stage 1 optimizer ready")
print(f"Epochs        : {EPOCHS_STAGE1}")
print(f"LR            : {LR_STAGE1}")
print(f"Weight decay  : {WEIGHT_DECAY}")
print(f"Loss          : CrossEntropy + Label Smoothing 0.1")

Stage 1 optimizer ready
Epochs        : 3
LR            : 0.001
Weight decay  : 0.0001
Loss          : CrossEntropy + Label Smoothing 0.1


In [10]:
# ==========================================================
# Cell 10 : Train / Validate One Epoch
# ==========================================================

def train_one_epoch(model, loader, optimizer, criterion, device, accum_steps=2):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()

    for step, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.to(device)

        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accum_steps * images.size(0)

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


print("Training and validation functions ready")

Training and validation functions ready


In [12]:
# ==========================================================
# Update AMP (PyTorch 2.x)
# ==========================================================

from torch.amp import autocast, GradScaler

# Recreate scaler using new API
scaler = GradScaler("cuda")

print("✅ Using new PyTorch AMP API")

✅ Using new PyTorch AMP API


In [13]:
# ==========================================================
# Updated Train / Validate Functions
# ==========================================================

def train_one_epoch(model, loader, optimizer, criterion, device, accum_steps=2):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    optimizer.zero_grad()

    for step, (images, labels) in enumerate(loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type="cuda"):
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if ((step + 1) % accum_steps == 0) or ((step + 1) == len(loader)):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accum_steps * images.size(0)

        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast(device_type="cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


print("✅ Updated training functions with new AMP API")

✅ Updated training functions with new AMP API


In [16]:
# ==========================================================
# Fix : Safe DataLoaders for Colab
# ==========================================================

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("✅ Safe DataLoaders ready")
print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

✅ Safe DataLoaders ready
Train batches: 1945
Test batches : 501


In [17]:
# ==========================================================
# Cell 11 : Stage 1 Training
# ==========================================================

best_acc = 0.0
stage1_history = []

for epoch in range(EPOCHS_STAGE1):

    start = time.time()

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        accum_steps=2
    )

    val_loss, val_acc = validate_one_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device
    )

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    elapsed = (time.time() - start) / 60

    stage1_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr,
        "time_min": elapsed
    })

    print(f"\nV6 Stage 1 Epoch {epoch+1}/{EPOCHS_STAGE1}")
    print("-" * 45)
    print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc:.4f}")
    print(f"LR         : {current_lr:.8f}")
    print(f"Time       : {elapsed:.2f} min")

    if val_acc > best_acc:
        best_acc = val_acc

        torch.save(model.state_dict(), "V6_Stage1_best.pth")

        print(f"✅ Best model saved! Val Acc: {best_acc:.4f}")

print("\nStage 1 completed.")
print(f"Best Validation Accuracy: {best_acc:.4f}")

stage1_df = pd.DataFrame(stage1_history)
stage1_df


V6 Stage 1 Epoch 1/3
---------------------------------------------
Train Loss : 2.4623 | Train Acc : 0.3220
Val Loss   : 2.3397 | Val Acc   : 0.3778
LR         : 0.00075000
Time       : 9.21 min
✅ Best model saved! Val Acc: 0.3778

V6 Stage 1 Epoch 2/3
---------------------------------------------
Train Loss : 2.2691 | Train Acc : 0.3887
Val Loss   : 2.2388 | Val Acc   : 0.4033
LR         : 0.00025000
Time       : 9.11 min
✅ Best model saved! Val Acc: 0.4033

V6 Stage 1 Epoch 3/3
---------------------------------------------
Train Loss : 2.1830 | Train Acc : 0.4199
Val Loss   : 2.2007 | Val Acc   : 0.4253
LR         : 0.00000000
Time       : 8.99 min
✅ Best model saved! Val Acc: 0.4253

Stage 1 completed.
Best Validation Accuracy: 0.4253


NameError: name 'pd' is not defined

In [18]:
import pandas as pd

stage1_df = pd.DataFrame(stage1_history)
stage1_df

,epoch,train_loss,train_acc,val_loss,val_acc,lr,time_min
0,1,2.462273,0.322042,2.339660,0.377811,0.00075,9.212343
1,2,2.269134,0.388700,2.238800,0.403298,0.00025,9.107548
2,3,2.183045,0.419875,2.200691,0.425287,0.00000,8.988366


In [19]:
# ==========================================================
# Cell 12 : Stage 2 Setup
# ==========================================================

# Load best Stage 1 weights
model.load_state_dict(torch.load("V6_Stage1_best.pth", map_location=device))

# Unfreeze entire model
for param in model.parameters():
    param.requires_grad = True

print("✅ Stage 1 weights loaded.")
print("✅ Entire ConvNeXt-Tiny unfrozen.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable Parameters : {trainable:,}")
print(f"Total Parameters     : {total:,}")

✅ Stage 1 weights loaded.
✅ Entire ConvNeXt-Tiny unfrozen.
Trainable Parameters : 27,837,815
Total Parameters     : 27,837,815


In [20]:
# ==========================================================
# Cell 13 : Stage 2 Optimizer + Scheduler
# ==========================================================

EPOCHS_STAGE2 = 10
LR_STAGE2 = 1e-4
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR_STAGE2,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS_STAGE2
)

best_acc = 0.0
stage2_history = []

print("✅ Stage 2 optimizer ready")
print(f"Epochs       : {EPOCHS_STAGE2}")
print(f"LR           : {LR_STAGE2}")
print(f"Weight decay : {WEIGHT_DECAY}")
print("Loss         : CrossEntropy + Label Smoothing 0.1")

✅ Stage 2 optimizer ready
Epochs       : 10
LR           : 0.0001
Weight decay : 0.0001
Loss         : CrossEntropy + Label Smoothing 0.1


In [1]:
# ==========================================================
# Cell 14 : Stage 2 Fine-Tuning
# ==========================================================

best_acc = 0.0
stage2_history = []

for epoch in range(EPOCHS_STAGE2):

    start = time.time()

    train_loss, train_acc = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        accum_steps=2
    )

    val_loss, val_acc = validate_one_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device
    )

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    elapsed = (time.time() - start) / 60

    stage2_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr,
        "time_min": elapsed
    })

    print(f"\nV6 Stage 2 Epoch {epoch+1}/{EPOCHS_STAGE2}")
    print("-" * 45)
    print(f"Train Loss : {train_loss:.4f} | Train Acc : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f} | Val Acc   : {val_acc:.4f}")
    print(f"LR         : {current_lr:.8f}")
    print(f"Time       : {elapsed:.2f} min")

    if val_acc > best_acc:
        best_acc = val_acc

        torch.save(model.state_dict(), "V6_Stage2_best.pth")

        print(f"✅ Best model saved! Val Acc: {best_acc:.4f}")

print("\nStage 2 completed.")
print(f"Best Validation Accuracy: {best_acc:.4f}")

# Optional history table
try:
    stage2_df = pd.DataFrame(stage2_history)
    display(stage2_df)
except:
    print("stage2_history saved, but pandas/display not available.")

NameError: name 'EPOCHS_STAGE2' is not defined

In [2]:
import os

print(os.path.exists("V6_Stage2_best.pth"))

False


In [3]:
import os

print("Current directory:", os.getcwd())
print("\nFiles:")
for f in os.listdir():
    if f.endswith(".pth"):
        print(f)

Current directory: /content

Files:


In [4]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/SkinDisease_V6"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving checkpoints to:", SAVE_DIR)

Mounted at /content/drive
Saving checkpoints to: /content/drive/MyDrive/SkinDisease_V6


In [5]:
torch.save(model.state_dict(), f"{SAVE_DIR}/V6_Stage1_best.pth")

NameError: name 'torch' is not defined

In [6]:
print("torch" in globals())
print("model" in globals())
print("SAVE_DIR" in globals())

False
False
True
